In [ ]:
import sqlite3
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from category_encoders import OneHotEncoder
from IPython.display import VimeoVideo
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.utils.validation import check_is_fitted

warnings.simplefilter(action="ignore", category=FutureWarning)


In [ ]:
VimeoVideo("665414155", h="c8a3e81a05", width=600)


In [ ]:
%load_ext sql
%sql sqlite:///../nepal.sqlite


In [ ]:
VimeoVideo("665415362", h="f677c48c46", width=600)


In [ ]:
%%sql

SELECT * 
FROM household_demographics


In [ ]:
%%sql
SELECT count(*) 
FROM household_demographics


In [ ]:
VimeoVideo("665415378", h="aa2b99493e", width=600)


In [ ]:
%%sql

SELECT * 
FROM id_map
LIMIT 5


In [ ]:
VimeoVideo("665415406", h="46a990c8f7", width=600)


In [ ]:
%%sql

SELECT h.*,
       s.*,
       i.vdcmun_id,
       d.damage_grade

FROM household_demographics AS h
JOIN id_map as i ON i.household_id = h.household_id
JOIN building_structure as s ON i.building_id = s.building_id
JOIN building_damage as d ON i.building_id = d.building_id
WHERE district_id = 4
LIMIT 5

/*
    step-1: 
    SELECT h.*
    FROM household_demographics AS h
    LIMIT 5

    Step-2: 
    SELECT h.*,
    i.vdcmun_id
    FROM household_demographics AS h
    JOIN id_map AS i ON i.household_id = h.household_id
    LIMIT 5

    Step:3
    SELECT h.*,
    s.*,
    i.vdcmun_id

FROM household_demographics AS h
JOIN id_map AS i ON i.household_id = h.household_id
JOIN building_structure AS s ON i.building_id = s.building_id
LIMIT 5
*/


In [ ]:
def wrangle(db_path):
    # Connect to database
    conn = sqlite3.connect(db_path)

    # Construct query
   
    query = """
    SELECT h.*,
       s.*,
       i.vdcmun_id,
       d.damage_grade

    FROM household_demographics AS h
    JOIN id_map as i ON i.household_id = h.household_id
    JOIN building_structure as s ON i.building_id = s.building_id
    JOIN building_damage as d ON i.building_id = d.building_id
    WHERE district_id = 4
    """
    
    
    # Read query results into DataFrame
    df = pd.read_sql(query, conn, index_col="household_id")

    # Identify leaky columns
    drop_cols = [col for col in df.columns if "post_eq" in col]

    # Add high-cardinality / redundant column
    drop_cols.append("building_id")

    # Create binary target column
    df["damage_grade"] = df["damage_grade"].str[-1].astype(int)
    df["severe_damage"] = (df["damage_grade"] > 3).astype(int)

    # Drop old target
    drop_cols.append("damage_grade")

    # Drop multicollinearity column
    drop_cols.append("count_floors_pre_eq")
  
    # Group caste column
    top_10 = df["caste_household"].value_counts().head(10).index

    df["caste_household"] = df["caste_household"].apply(
    lambda c: c if c in top_10 else "Other"
    )
    # Drop columns
    df.drop(columns=drop_cols, inplace=True)

    return df


In [ ]:
VimeoVideo("665415443", h="ca27a7ebfc", width=600)


In [ ]:
df = wrangle("../nepal.sqlite")
df.head()


In [ ]:
# Check your work
assert df.shape == (75883, 20), f"`df` should have shape (75883, 20), not {df.shape}"


In [ ]:
VimeoVideo("665415463", h="86c306199f", width=600)


In [ ]:
# Check for high- and low-cardinality categorical features
df.select_dtypes("object").nunique()


In [ ]:
VimeoVideo("665415472", h="1142d69e4a", width=600)


In [ ]:
df['caste_household'].nunique()


In [ ]:
# Check your work
assert (
    df["caste_household"].nunique() == 11
), f"The `'caste_household'` column should only have 11 unique values, not {df['caste_household'].nunique()}."


In [ ]:
VimeoVideo("665415515", h="defc252edd", width=600)


In [ ]:
target = "severe_damage"
X = df.drop(columns=[target, "vdcmun_id"])
y = df[target]


In [ ]:
# Check your work
assert X.shape == (75883, 18), f"The shape of `X` should be (75883, 18), not {X.shape}."
assert "vdcmun_id" not in X.columns, "There should be no `'vdcmun_id'` column in `X`."
assert y.shape == (75883,), f"The shape of `y` should be (75883,), not {y.shape}."


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)


In [ ]:
# Check your work
assert X_train.shape == (
    60706,
    18,
), f"The shape of `X_train` should be (60706, 18), not {X_train.shape}."
assert y_train.shape == (
    60706,
), f"The shape of `y_train` should be (60706,), not {y_train.shape}."
assert X_test.shape == (
    15177,
    18,
), f"The shape of `X_test` should be (15177, 18), not {X_test.shape}."
assert y_test.shape == (
    15177,
), f"The shape of `y_test` should be (15177,), not {y_test.shape}."


In [ ]:
y_train.value_counts(normalize=True)


In [ ]:
acc_baseline = y_train.value_counts(normalize=True).max()
print("Baseline Accuracy:", round(acc_baseline, 2))


In [ ]:
model_lr = make_pipeline(
    OneHotEncoder(use_cat_names=True),
    LogisticRegression(max_iter=1000)
)

model_lr.fit(X_train, y_train)


In [ ]:
# Check your work
assert isinstance(
    model_lr, Pipeline
), f"`model_lr` should be a Pipeline, not type {type(model_lr)}."
assert isinstance(
    model_lr[0], OneHotEncoder
), f"The first step in your Pipeline should be a OneHotEncoder, not type {type(model_lr[0])}."
assert isinstance(
    model_lr[-1], LogisticRegression
), f"The last step in your Pipeline should be LogisticRegression, not type {type(model_lr[-1])}."
check_is_fitted(model_lr)


In [ ]:
acc_train = accuracy_score(y_train, model_lr.predict(X_train))
acc_test = model_lr.score(X_test, y_test)

print("LR Training Accuracy:", acc_train)
print("LR Validation Accuracy:", acc_test)


In [ ]:
VimeoVideo("665415532", h="00440f76a9", width=600)


In [ ]:
features = model_lr.named_steps["onehotencoder"].get_feature_names_out()
importances = model_lr.named_steps["logisticregression"].coef_[0]
feat_imp = pd.Series(np.exp(importances), index=features).sort_values()
feat_imp.head()


In [ ]:
VimeoVideo("665415552", h="5b2383ccf8", width=600)


In [ ]:
feat_imp.tail(10).plot(kind="barh")
plt.xlabel("Odds ratio")


In [ ]:
VimeoVideo("665415581", h="d15477e14d", width=600)


In [ ]:
feat_imp.head(10).plot(kind="barh")
plt.xlabel("Odds ratio")


In [ ]:
VimeoVideo("665415631", h="90ba264392", width=600)


In [ ]:
damage_by_vdcmun = (
    df.groupby("vdcmun_id")["severe_damage"].mean().sort_values(ascending=False)).to_frame()
damage_by_vdcmun


In [ ]:
# Check your work
assert isinstance(
    damage_by_vdcmun, pd.DataFrame
), f"`damage_by_vdcmun` should be a Series, not type {type(damage_by_vdcmun)}."
assert damage_by_vdcmun.shape == (
    11,
    1,
), f"`damage_by_vdcmun` should be shape (11,1), not {damage_by_vdcmun.shape}."


In [ ]:
VimeoVideo("665415651", h="9b5244dec1", width=600)


In [ ]:
# Plot line
plt.plot(damage_by_vdcmun.values, color="gray")
plt.xticks(range(len(damage_by_vdcmun)), labels=damage_by_vdcmun.index )
plt.yticks(np.arange(0.0, 1.1, 0.2))
plt.xlabel("Municipality ID")
plt.ylabel("% of Total Households")
plt.title("Severe Damage by Municipality");


In [ ]:
VimeoVideo("665415693", h="fb2e54aa04", width=600)


In [ ]:
damage_by_vdcmun["Gurung"] = (
    df[df["caste_household"] == "Gurung"].groupby("vdcmun_id")["severe_damage"].count()
    / df.groupby("vdcmun_id")["severe_damage"].count()
)
damage_by_vdcmun


In [ ]:
VimeoVideo("665415707", h="9b29c23434", width=600)


In [ ]:
damage_by_vdcmun["Kumal"] = (
    df[df["caste_household"]=="Kumal"].groupby("vdcmun_id")["severe_damage"].count()
    /
    df.groupby("vdcmun_id")["severe_damage"].count()
).fillna(0)
damage_by_vdcmun


In [ ]:
VimeoVideo("665415729", h="8d0712c306", width=600)


In [ ]:
# Plot line
damage_by_vdcmun.drop(columns="severe_damage").plot(kind="bar", stacked=True)
plt.plot(damage_by_vdcmun["severe_damage"].values, color="gray")
plt.xticks(range(len(damage_by_vdcmun)), labels=damage_by_vdcmun.index )
plt.yticks(np.arange(0.0, 1.1, 0.2))
plt.xlabel("Municipality ID")
plt.ylabel("% of Total Households")
plt.title("Severe Damage by Municipality");
